In [1]:
!pip install geopy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [geopy]32m1/2 [geopy]


In [2]:
from geopy.geocoders import Nominatim

In [3]:
app = Nominatim(user_agent='streetvibes')

In [6]:
app.geocode('Shoot-Up Hill, London').raw

{'place_id': 259407844,
 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright',
 'osm_type': 'way',
 'osm_id': 1071976817,
 'lat': '51.5490063',
 'lon': '-0.2058739',
 'class': 'historic',
 'type': 'roman_road',
 'place_rank': 30,
 'importance': 9.307927061870783e-05,
 'addresstype': 'historic',
 'name': 'Shoot-up Hill',
 'display_name': 'Shoot-up Hill, Kilburn, London Borough of Camden, Greater London, England, NW2 3QL, United Kingdom',
 'boundingbox': ['51.5486916', '51.5493176', '-0.2062117', '-0.2055391']}

In [7]:
app.geocode('Canal Street Manchester').raw

{'place_id': 256796930,
 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright',
 'osm_type': 'way',
 'osm_id': 553519695,
 'lat': '53.4775865',
 'lon': '-2.2359908',
 'class': 'highway',
 'type': 'pedestrian',
 'place_rank': 26,
 'importance': 0.05341150606546117,
 'addresstype': 'road',
 'name': 'Canal Street',
 'display_name': 'Canal Street, Gay Village, City Centre, Manchester, Greater Manchester, England, M1 3HN, United Kingdom',
 'boundingbox': ['53.4769267', '53.4782275', '-2.2370428', '-2.2349269']}

In [9]:
app.geocode('AL1 1JJ').raw

{'place_id': 350458336,
 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright',
 'lat': '51.7444800',
 'lon': '-0.3222700',
 'class': 'place',
 'type': 'postcode',
 'place_rank': 25,
 'importance': 0.06667666666666666,
 'addresstype': 'postcode',
 'name': 'AL1 1JJ',
 'display_name': 'AL1 1JJ, St Albans, Hertfordshire, England, United Kingdom',
 'boundingbox': ['51.7424800', '51.7464800', '-0.3242700', '-0.3202700']}

In [11]:
app.geocode('british library').raw

{'place_id': 259333508,
 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright',
 'osm_type': 'way',
 'osm_id': 4680891,
 'lat': '51.5299119',
 'lon': '-0.1276918',
 'class': 'amenity',
 'type': 'library',
 'place_rank': 30,
 'importance': 0.6214365190760418,
 'addresstype': 'amenity',
 'name': 'British Library',
 'display_name': 'British Library, 96, Euston Road, Somers Town, London Borough of Camden, Greater London, England, NW1 2DB, United Kingdom',
 'boundingbox': ['51.5289846', '51.5310199', '-0.1289210', '-0.1264963']}

In [72]:
def enforce_mapillary_bbox_limit(min_lon, min_lat, max_lon, max_lat, max_span=0.099):
    """
    Ensure the bbox spans do not exceed Mapillary's <= 0.01° requirement.
    If spans are larger, shrink the box around its center.
    """
    lat_span = max_lat - min_lat
    lon_span = max_lon - min_lon

    def clamp(center, span):
        half = min(span / 2, max_span / 2)
        return center - half, center + half

    # Compute centers
    lat_c = (min_lat + max_lat) / 2
    lon_c = (min_lon + max_lon) / 2

    # Clamp spans
    min_lat, max_lat = clamp(lat_c, lat_span)
    min_lon, max_lon = clamp(lon_c, lon_span)

    return min_lon, min_lat, max_lon, max_lat

In [73]:
import os
import requests


def bbox_from_geopy_raw(geo_raw, buffer_deg=0.001):
    """
    Convert a geopy .raw result into a Mapillary bbox: (min_lon, min_lat, max_lon, max_lat).
    If bbox is missing or degenerate (point), buffer around lat/lon by buffer_deg.

    For mapillary, order: left, bottom, right, top, aka
        minLon, minLat, maxLon, maxLat).
        NOTE: The bbox area must be smaller than 0.01 degrees square.
    See https://www.mapillary.com/developer/api-documentation/
    """
    
    lat = float(geo_raw['lat'])
    lon = float(geo_raw['lon'])
    bbox = geo_raw.get('boundingbox')

    if bbox:
        south, north, west, east = map(float, bbox)

    # Enforce Mapillary constraint
    min_lon, min_lat, max_lon, max_lat = enforce_mapillary_bbox_limit(
        west, south, east, north
    )
    return min_lon, min_lat, max_lon, max_lat
    
    #return west, south, east, north

    #return None

    """
    
    if bbox:
        # geopy/Nominatim: [south, north, west, east]
        south, north, west, east = map(float, bbox)

        if abs(north - south) < buffer_deg:
            south -= buffer_deg/2.0
            north += buffer_deg/2.0

        if abs(east - west) < buffer_deg:
            west -= buffer_deg/2.0
            east += buffer_deg/2.0
    else:
        # No bbox: build one from point + buffer
        south = lat - buffer_deg
        north = lat + buffer_deg
        west = lon - buffer_deg
        east = lon + buffer_deg

    print(west, south, east, north)
    return west, south, east, north
    """

In [74]:
def download_mapillary_images_for_geopy_raw(
    geo_raw,
    access_token,
    out_dir='mapillary_images',
    limit=5,
    buffer_deg=0.001
):
    """
    Given a geopy .raw result and a Mapillary access token, download up to `limit`
    images that intersect the bounding box (or a buffered point bbox).

    Returns a list of local file paths.
    """
    os.makedirs(out_dir, exist_ok=True)

    min_lon, min_lat, max_lon, max_lat = bbox_from_geopy_raw(geo_raw, buffer_deg=buffer_deg)

    params = {
        'access_token': access_token,
        'fields': 'id,thumb_1024_url,captured_at',
        'bbox': f'{min_lon},{min_lat},{max_lon},{max_lat}',
        'limit': limit,
    }

    r = requests.get('https://graph.mapillary.com/images', params=params, timeout=10)
    r.raise_for_status()
    data = r.json().get('data', [])

    print(r.text)

    # If you want strictly “most recent”, sort by captured_at (if present)
    #data.sort(key=lambda x: x.get('captured_at', 0), reverse=True)
    #data = data[:limit]

    downloaded_paths = []

    for img in data:
        img_id = img['id']
        url = img.get('thumb_1024_url')
        if not url:
            continue

        resp = requests.get(url, timeout=10)
        resp.raise_for_status()

        out_path = os.path.join(out_dir, f'{img_id}.jpg')
        with open(out_path, 'wb') as f:
            f.write(resp.content)

        downloaded_paths.append(out_path)

    return downloaded_paths

In [75]:
import re

def slugify(s):
    s = s.lower()
    s = re.sub(r'[^a-z0-9]+', '-', s)      # replace non-alphanumerics with dashes
    s = re.sub(r'-+', '-', s).strip('-')   # collapse and trim dashes
    return s

In [77]:
from geopy.geocoders import Nominatim
import json

geolocator = Nominatim(user_agent='streetvibes')
TOKEN = 'MLY|23917134374545382|7b84681475ad296314c2f7fb1b6ea991'

query = 'oxford street london'
loc = geolocator.geocode(query).raw

print(json.dumps(loc, indent=2))

paths = download_mapillary_images_for_geopy_raw(
    loc,
    access_token=TOKEN,
    out_dir=slugify(query),
    limit=5,
    buffer_deg=0.001  # ~100 m
)

print(paths)

{
  "place_id": 260004443,
  "licence": "Data \u00a9 OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright",
  "osm_type": "way",
  "osm_id": 204726687,
  "lat": "51.5143256",
  "lon": "-0.1496938",
  "class": "highway",
  "type": "primary",
  "place_rank": 26,
  "importance": 0.5234054631758154,
  "addresstype": "road",
  "name": "Oxford Street",
  "display_name": "Oxford Street, East Marylebone, Mayfair, City of Westminster, Greater London, England, W1C 1JN, United Kingdom",
  "boundingbox": [
    "51.5142692",
    "51.5143915",
    "-0.1502156",
    "-0.1491731"
  ]
}
{"data":[{"id":"101061209351399","thumb_1024_url":"https:\/\/scontent-lhr6-2.xx.fbcdn.net\/m1\/v\/t6\/An-lcpS0bEULuflULadcX1Xr2LqZZE1Wbb8J1VrmaICyUlVC9zIttAtiXVv-H0an60eZFKgFLGseqMVjNlG8wpit79UUzSYC-0PtBsygwaMz35NPnU815K5H9HcJTp3urpetc6ZZ2622MKsVg8UUng?stp=s1024x768&edm=AOnQwmMEAAAA&_nc_gid=P4oNOQ0An0LcrsuDhX61qw&_nc_oc=Adk3TNL67RVVBtc3RFekwPdD1fe5_flACSckEUIq0ruR88o3oF-uixWJXGVVNza-VMQ&ccb=10-5&oh=00_AfnXCRAH